# Etapa 1 - Setup do Ambiente

Nessa etapa o objetivo vai ser instalar as dependências e fazer a configuração do ambiente

In [1]:
# Nesse primeiro momento, vamos instalar as bibliotecas necessárias para o pipeline.
# 'great_expectatios' é uma das principais libs de qualidade de dados.
# colorama e tabulate são auxiliares para deixar o output do notebook mais legível durante a execução.

!pip install great_expectations pandas requests colorama tabulate --quiet

print("Dependências instaladas com sucesso!")

Dependências instaladas com sucesso!


In [2]:
# Imports e configurações globais
import pandas as pd
import numpy as np
import os
import hashlib
from datetime import datetime, timezone
from colorama import Fore, Style, init
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

init(autoreset=True)

# Configuração de caminhos para simular um Data Lake local
BASE_PATH = "./data_lake"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"
QUALITY_PATH = f"{BASE_PATH}/quality_reports"

for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUALITY_PATH]:
  os.makedirs(path, exist_ok=True)

# Timestamp de ingestão é gerado uma vez aqui e reutilizado em todas as etapas
INGESTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print(f"{Fore.GREEN} Ambiente configurado!")
print(f"{Fore.CYAN} Estrutura do Data Lake:")
print(f"    {BASE_PATH}/")
print(f"    ├── bronze/   -> Dados brutos (raw)")
print(f"    ├── silver/  -> Dados limpos e normalizados")
print(f"    ├── gold/    -> Dados agregados para consumo")
print(f"    └── quality_reports/ -> Relatórios de qualidade")
print(f"\n{Fore.YELLOW} Timestamp de ingestão: {INGESTION_TIMESTAMP}")

 Ambiente configurado!
 Estrutura do Data Lake:
    ./data_lake/
    ├── bronze/   -> Dados brutos (raw)
    ├── silver/  -> Dados limpos e normalizados
    ├── gold/    -> Dados agregados para consumo
    └── quality_reports/ -> Relatórios de qualidade

 Timestamp de ingestão: 20261019_000110


# Etapa 2 - Camada BRONZE: Ingestão de Dados

A camada Bronze armazena os dados exatamente como vieram da fonte — sem nenhuma transformação. Isso garante rastreabilidade e a possibilidade de reprocessamento.

In [3]:
# Optamos por consultar diretamente o BigQuery em vez de arquivos CSV 
# locais para garantir a governança, integridade e escalabilidade dos dados corporativos,
# evitando problemas de versionamento e limites de tamanho de arquivos estáticos.

In [ ]:
# Instalamos o BigQuery Storage e o PyArrow para baixar os dados em formato binário 
# (via Apache Arrow) em vez de texto puro (CSV). Isso evita a conversão linha por linha,
# acelerando drasticamente a transferência de grandes volumes para o DataFrame do Pandas.
!pip install google-cloud-bigquery db-dtypes pandas pyarrow

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./credenciais_gcp.json"

client = bigquery.Client()

print("Conexão com o BigQuery configurada com sucesso!")

Conexão com o BigQuery configurada com sucesso!


In [6]:
from google.cloud import bigquery
import pandas as pd
import hashlib
import os
from colorama import Fore

def ingest_from_bigquery(table_id: str, entity_name: str) -> pd.DataFrame:
    """
    Ingere dados de uma tabela do BigQuery (Base dos Dados) e salva na camada Bronze.
    Caso o arquivo já exista localmente e haja falha de autenticação/conexão no BigQuery,
    carrega os dados do cache local de forma transparente.
    """
    print(f"{Fore.CYAN} Iniciando ingestão da entidade [{entity_name}]...")
    print(f"    Tabela fonte: {table_id}")

    # Nome de arquivo fixo e padronizado na Bronze (sem timestamp no nome para evitar duplicação)
    output_file = f"{BRONZE_PATH}/{entity_name}_raw.parquet"

    try:
        # 1. Monta e executa a consulta SQL no BigQuery
        query = f"SELECT * FROM `{table_id}`"
        df = client.query(query).to_dataframe(create_bqstorage_client=True)

        # 2. Adiciona metadados de ingestão
        df['_ingestion_timestamp'] = INGESTION_TIMESTAMP
        df['_source_table']        = table_id
        df['_source_system']       = 'basedosdados_bigquery'
        
        # 3. Hash de rastreabilidade (calculado de forma vetorizada/eficiente)
        df['_record_hash'] = df.drop(
            columns=['_ingestion_timestamp', '_source_table', '_source_system'],
            errors='ignore'
        ).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

        # 4. Salva/Sobrescreve o arquivo Parquet fixo na camada Bronze
        df.to_parquet(output_file, index=False)

        print(f"{Fore.GREEN}    {len(df)} registros ingeridos do BigQuery com sucesso.")
        print(f"    Salvo em: {output_file}")
        return df

    except Exception as e:
        print(f"{Fore.YELLOW}    Aviso/Falha na consulta ao BigQuery: {e}")
        
        # Fallback: Se o arquivo já existe localmente, lê do disco para não travar a execução
        if os.path.exists(output_file):
            print(f"{Fore.GREEN}    [FALLBACK] Carregando dados locais pré-existentes da Bronze: {output_file}")
            df = pd.read_parquet(output_file)
            return df
        else:
            print(f"{Fore.RED}    ERRO: Não foi possível conectar ao BigQuery e o arquivo local não existe.")
            raise

print(f"{Fore.GREEN} Função de ingestão do BigQuery corrigida com sucesso!")

 Função de ingestão do BigQuery corrigida com sucesso!


In [7]:
# ============================================================
# ETAPA 2.2 — INGESTÃO MULTI-ENTIDADE
# ============================================================
# Ingerimos as entidades do projeto de avaliação da alfabetização:
# - dicionario                → dicionário de dados / variáveis
# - municipio                → indicadores consolidados por município
# - uf                       → indicadores consolidados por UF
# - meta_alfabetizacao_brasil  → trajetória e metas nacionais
# - meta_alfabetizacao_uf      → trajetória e metas por UF
# - meta_alfabetizacao_municipio → trajetória e metas por município

print(f"{Fore.YELLOW}{'='*60}")
print(f"{Fore.YELLOW}  INICIANDO PIPELINE DE INGESTÃO (BIGQUERY)")
print(f"{Fore.YELLOW}{'='*60}\n")

# Dicionário com todas as entidades a ingerir do BigQuery (Base dos Dados)
entities_bigquery = {
    'municipio':                    'basedosdados.br_inep_avaliacao_alfabetizacao.municipio',
    'uf':                           'basedosdados.br_inep_avaliacao_alfabetizacao.uf',
    'meta_alfabetizacao_brasil':    'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil',
    'meta_alfabetizacao_uf':        'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf',
    'meta_alfabetizacao_municipio': 'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio',
    'dicionario':                   'basedosdados.br_inep_avaliacao_alfabetizacao.dicionario',
    'alunos':                       'basedosdados.br_inep_avaliacao_alfabetizacao.alunos'  
}

bronze_dfs = {}

for entity, table_id in entities_bigquery.items():
    bronze_dfs[entity] = ingest_from_bigquery(table_id, entity)
    print()

print(f"{Fore.GREEN}{'='*60}")
print(f"{Fore.GREEN}  INGESTÃO CONCLUÍDA")
print(f"{Fore.GREEN}   Total de entidades: {len(bronze_dfs)}")
print(f"{Fore.GREEN}   Total de registros: {sum(len(df) for df in bronze_dfs.values())}")
print(f"{Fore.GREEN}{'='*60}")

  INICIANDO PIPELINE DE INGESTÃO (BIGQUERY)

 Iniciando ingestão da entidade [municipio]...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.municipio
    23995 registros ingeridos do BigQuery com sucesso.
    Salvo em: ./data_lake/bronze/municipio_raw.parquet

 Iniciando ingestão da entidade [uf]...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.uf
    145 registros ingeridos do BigQuery com sucesso.
    Salvo em: ./data_lake/bronze/uf_raw.parquet

 Iniciando ingestão da entidade [meta_alfabetizacao_brasil]...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
    3 registros ingeridos do BigQuery com sucesso.
    Salvo em: ./data_lake/bronze/meta_alfabetizacao_brasil_raw.parquet

 Iniciando ingestão da entidade [meta_alfabetizacao_uf]...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
    81 registros ingeridos do BigQuery com sucesso.
    Salvo em: ./data_lake/bronze/meta_alfabet

In [8]:
# ============================================================
# ETAPA 2.3 — INSPEÇÃO DOS DADOS BRUTOS
# ============================================================
# Sempre inspecionar os dados antes de processá-los!

print(f"{Fore.YELLOW} PRÉVIA DOS DADOS BRUTOS (Bronze)\n")

for entity, df in bronze_dfs.items():
    print(f"{Fore.CYAN}{'─'*50}")
    print(f"{Fore.CYAN}  Entidade: {entity.upper()}")
    print(f"   Shape (Linhas, Colunas): {df.shape} | Nulos totais: {df.isnull().sum().sum()}")
    print(f"   Tipos: { {col: str(dtype) for col, dtype in df.dtypes.items()} }")
    print()
    display(df.head(3))
    print()

 PRÉVIA DOS DADOS BRUTOS (Bronze)

──────────────────────────────────────────────────
  Entidade: MUNICIPIO
   Shape (Linhas, Colunas): (23995, 19) | Nulos totais: 103923
   Tipos: {'ano': 'Int64', 'id_municipio': 'str', 'serie': 'str', 'rede': 'str', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'str', '_source_table': 'str', '_source_system': 'str', '_record_hash': 'str'}



,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,d1b6df4e80959ce1a5082a62c2f1db0c
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,4cda7a8d76f3afa7a02c2237a0868043
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,5b53ac3da3f4a8d4c8a16760b6a46b74



──────────────────────────────────────────────────
  Entidade: UF
   Shape (Linhas, Colunas): (145, 19) | Nulos totais: 630
   Tipos: {'ano': 'Int64', 'sigla_uf': 'str', 'serie': 'str', 'rede': 'str', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'str', '_source_table': 'str', '_source_system': 'str', '_record_hash': 'str'}



,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,42ee6a943b5c1b9714c8e978c57f248b
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,eca51cb864fb5c8429b4b562aa79eee6
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,586978b39f93e4d089e198ec3945236a



──────────────────────────────────────────────────
  Entidade: META_ALFABETIZACAO_BRASIL
   Shape (Linhas, Colunas): (3, 15) | Nulos totais: 0
   Tipos: {'ano': 'Int64', 'rede': 'str', 'taxa_alfabetizacao': 'float64', 'meta_alfabetizacao_2024': 'float64', 'meta_alfabetizacao_2025': 'float64', 'meta_alfabetizacao_2026': 'float64', 'meta_alfabetizacao_2027': 'float64', 'meta_alfabetizacao_2028': 'float64', 'meta_alfabetizacao_2029': 'float64', 'meta_alfabetizacao_2030': 'float64', 'percentual_participacao': 'float64', '_ingestion_timestamp': 'str', '_source_table': 'str', '_source_system': 'str', '_record_hash': 'str'}



...

,ano,id_municipio,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,4301750,Municipal,NaN,NaN,14.05,23.65,37.0,52.68,67.85,80.0,<NA>,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,d599e2a6152fe5e49952a7f9c07e7e02
1,2024,4301750,Municipal,4.40,NaN,14.05,23.65,37.0,52.68,67.85,80.0,0,92.59,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,f8c0b9945b3bb6fe09ea207f9311f87e
2,2024,2406908,Municipal,42.86,7.94,14.05,23.65,37.0,52.68,67.85,80.0,1,84.00,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,dd484f706727c46b70499c6b432f3f2d



──────────────────────────────────────────────────
  Entidade: DICIONARIO
   Shape (Linhas, Colunas): (27, 9) | Nulos totais: 0
   Tipos: {'id_tabela': 'str', 'nome_coluna': 'str', 'chave': 'str', 'cobertura_temporal': 'str', 'valor': 'str', '_ingestion_timestamp': 'str', '_source_table': 'str', '_source_system': 'str', '_record_hash': 'str'}



,id_tabela,nome_coluna,chave,cobertura_temporal,valor,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,alunos,alfabetizado,0,(1),Não,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,c0ed42532a60244b66c2ebd4d3abfbfc
1,alunos,alfabetizado,1,(1),Sim,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,4501a899a067f72a9b9bbbd54440c890
2,alunos,preenchimento_caderno,0,(1),Prova não preenchida,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,f6b07fc63b2bc5cdd8ae5de8092dce75



──────────────────────────────────────────────────
  Entidade: ALUNOS
   Shape (Linhas, Colunas): (3867999, 16) | Nulos totais: 1026676
   Tipos: {'ano': 'Int64', 'id_municipio': 'str', 'id_escola': 'str', 'id_aluno': 'str', 'caderno': 'str', 'serie': 'str', 'rede': 'str', 'presenca': 'str', 'preenchimento_caderno': 'str', 'alfabetizado': 'str', 'proficiencia': 'float64', 'peso_aluno': 'float64', '_ingestion_timestamp': 'str', '_source_table': 'str', '_source_system': 'str', '_record_hash': 'str'}



,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,1502400,60001927,15039653,1,2,3,0,0,0,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,5f6da82a750fdd6e5d6a96540c3e60d7
1,2023,3205002,60023501,32007457,1,2,3,0,0,0,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,5e3716d6cfd3c372f787952488ee4e7c
2,2023,3303708,60023787,33149373,1,2,3,0,0,0,NaN,NaN,20261019_000110,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,1988cd04b7af1b4adbde7581edf83c42


# Etapa 3 - Camada SILVER: Limpeza e Normalização

In [9]:
# ============================================================
# ETAPA 3.1 — TRANSFORMAÇÃO: MUNICIPIO
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: MUNICIPIO\n")

# 1. Cópia do dataframe bronze
df_municipio_silver = bronze_dfs['municipio'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_municipio_silver)
df_municipio_silver = df_municipio_silver.drop_duplicates()
linhas_depois = len(df_municipio_silver)

print(f"Duplicatas removidas: {linhas_antes - linhas_depois}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_municipio_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_municipio_silver['id_municipio'] = df_municipio_silver['id_municipio'].astype(str)
df_municipio_silver['serie'] = df_municipio_silver['serie'].astype(str)
df_municipio_silver['rede'] = df_municipio_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao', 'media_portugues'] + [col for col in df_municipio_silver.columns if 'proporcao_aluno_nivel' in col]
for col in colunas_numericas:
    df_municipio_silver[col] = pd.to_numeric(df_municipio_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_municipio_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_municipio_silver['_record_hash'] = df_municipio_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/municipio_silver.parquet"
df_municipio_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade MUNICIPIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_municipio_silver)}")

  Processando entidade: MUNICIPIO

Duplicatas removidas: 0

Valores nulos por coluna:
ano                            0
id_municipio                   0
serie                          0
rede                           0
taxa_alfabetizacao             0
media_portugues                0
proporcao_aluno_nivel_0    11547
proporcao_aluno_nivel_1    11547
proporcao_aluno_nivel_2    11547
proporcao_aluno_nivel_3    11547
proporcao_aluno_nivel_4    11547
proporcao_aluno_nivel_5    11547
proporcao_aluno_nivel_6    11547
proporcao_aluno_nivel_7    11547
proporcao_aluno_nivel_8    11547
_ingestion_timestamp           0
_source_table                  0
_source_system                 0
_record_hash                   0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
id_municipio                   str
serie                          str
rede                           str
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64


In [10]:
# ============================================================
# ETAPA 3.2 — TRANSFORMAÇÃO: UF
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: UF\n")

# 1. Cópia do dataframe bronze
df_uf_silver = bronze_dfs['uf'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_uf_silver)
df_uf_silver = df_uf_silver.drop_duplicates()
linhas_depois = len(df_uf_silver)

print(f"Duplicatas removidas: {linhas_antes - linhas_depois}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_uf_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_uf_silver['sigla_uf'] = df_uf_silver['sigla_uf'].astype(str)
df_uf_silver['serie'] = df_uf_silver['serie'].astype(str)
df_uf_silver['rede'] = df_uf_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao', 'media_portugues'] + [col for col in df_uf_silver.columns if 'proporcao_aluno_nivel' in col]
for col in colunas_numericas:
    df_uf_silver[col] = pd.to_numeric(df_uf_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_uf_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_uf_silver['_record_hash'] = df_uf_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/uf_silver.parquet"
df_uf_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade UF processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_uf_silver)}")

  Processando entidade: UF

Duplicatas removidas: 0

Valores nulos por coluna:
ano                         0
sigla_uf                    0
serie                       0
rede                        0
taxa_alfabetizacao          0
media_portugues             0
proporcao_aluno_nivel_0    70
proporcao_aluno_nivel_1    70
proporcao_aluno_nivel_2    70
proporcao_aluno_nivel_3    70
proporcao_aluno_nivel_4    70
proporcao_aluno_nivel_5    70
proporcao_aluno_nivel_6    70
proporcao_aluno_nivel_7    70
proporcao_aluno_nivel_8    70
_ingestion_timestamp        0
_source_table               0
_source_system              0
_record_hash                0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
sigla_uf                       str
serie                          str
rede                           str
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64
proporcao_aluno_nivel_2    fl

In [11]:
# ============================================================
# ETAPA 3.3 — TRANSFORMAÇÃO: META_ALFABETIZACAO_BRASIL
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: META_ALFABETIZACAO_BRASIL\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_brasil_silver = bronze_dfs['meta_alfabetizacao_brasil'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_brasil_silver)
df_meta_alfabetizacao_brasil_silver = df_meta_alfabetizacao_brasil_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_brasil_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_brasil_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_brasil_silver['rede'] = df_meta_alfabetizacao_brasil_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_brasil_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_brasil_silver[col] = pd.to_numeric(df_meta_alfabetizacao_brasil_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_brasil_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_brasil_silver['_record_hash'] = df_meta_alfabetizacao_brasil_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_brasil_silver.parquet"
df_meta_alfabetizacao_brasil_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_BRASIL processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_brasil_silver)}")

 Processando entidade: META_ALFABETIZACAO_BRASIL

Duplicatas removidas: 0

Valores nulos por coluna:
ano                        0
rede                       0
taxa_alfabetizacao         0
meta_alfabetizacao_2024    0
meta_alfabetizacao_2025    0
meta_alfabetizacao_2026    0
meta_alfabetizacao_2027    0
meta_alfabetizacao_2028    0
meta_alfabetizacao_2029    0
meta_alfabetizacao_2030    0
percentual_participacao    0
_ingestion_timestamp       0
_source_table              0
_source_system             0
_record_hash               0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
rede                           str
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_participacao    float64
_ingestion_timestamp           str
_s

In [12]:
# ============================================================
# ETAPA 3.4 — TRANSFORMAÇÃO: META_ALFABETIZACAO_UF
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: META_ALFABETIZACAO_UF\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_uf_silver = bronze_dfs['meta_alfabetizacao_uf'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_uf_silver)
df_meta_alfabetizacao_uf_silver = df_meta_alfabetizacao_uf_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_uf_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_uf_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_uf_silver['sigla_uf'] = df_meta_alfabetizacao_uf_silver['sigla_uf'].astype(str)
df_meta_alfabetizacao_uf_silver['rede'] = df_meta_alfabetizacao_uf_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_uf_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_uf_silver[col] = pd.to_numeric(df_meta_alfabetizacao_uf_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_uf_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_uf_silver['_record_hash'] = df_meta_alfabetizacao_uf_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_uf_silver.parquet"
df_meta_alfabetizacao_uf_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_UF processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_uf_silver)}")

 Processando entidade: META_ALFABETIZACAO_UF

Duplicatas removidas: 0

Valores nulos por coluna:
ano                        0
sigla_uf                   0
rede                       0
taxa_alfabetizacao         4
meta_alfabetizacao_2024    9
meta_alfabetizacao_2025    3
meta_alfabetizacao_2026    2
meta_alfabetizacao_2027    2
meta_alfabetizacao_2028    2
meta_alfabetizacao_2029    2
meta_alfabetizacao_2030    2
percentual_participacao    4
_ingestion_timestamp       0
_source_table              0
_source_system             0
_record_hash               0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
sigla_uf                       str
rede                           str
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_p

In [13]:
# ============================================================
# ETAPA 3.5 — TRANSFORMAÇÃO: META_ALFABETIZACAO_MUNICIPIO
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: META_ALFABETIZACAO_MUNICIPIO\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_municipio_silver = bronze_dfs['meta_alfabetizacao_municipio'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_municipio_silver)
df_meta_alfabetizacao_municipio_silver = df_meta_alfabetizacao_municipio_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_municipio_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_municipio_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_municipio_silver['id_municipio'] = df_meta_alfabetizacao_municipio_silver['id_municipio'].astype(str)
df_meta_alfabetizacao_municipio_silver['rede'] = df_meta_alfabetizacao_municipio_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_municipio_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_municipio_silver[col] = pd.to_numeric(df_meta_alfabetizacao_municipio_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_municipio_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_municipio_silver['_record_hash'] = df_meta_alfabetizacao_municipio_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_municipio_silver.parquet"
df_meta_alfabetizacao_municipio_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_MUNICIPIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_municipio_silver)}")

 Processando entidade: META_ALFABETIZACAO_MUNICIPIO

Duplicatas removidas: 0

Valores nulos por coluna:
ano                          0
id_municipio                 0
rede                         0
taxa_alfabetizacao         120
meta_alfabetizacao_2024    240
meta_alfabetizacao_2025      0
meta_alfabetizacao_2026      0
meta_alfabetizacao_2027      0
meta_alfabetizacao_2028      0
meta_alfabetizacao_2029      0
meta_alfabetizacao_2030      0
nivel_alfabetizacao        120
percentual_participacao    120
_ingestion_timestamp         0
_source_table                0
_source_system               0
_record_hash                 0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
id_municipio                   str
rede                           str
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabet

In [14]:
# ============================================================
# ETAPA 3.6 — TRANSFORMAÇÃO: DICIONARIO
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: DICIONARIO\n")

# 1. Cópia do dataframe bronze
df_dicionario_silver = bronze_dfs['dicionario'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_dicionario_silver)
df_dicionario_silver = df_dicionario_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_dicionario_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_dicionario_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
colunas_texto = ['id_tabela', 'nome_coluna', 'chave', 'cobertura_temporal', 'valor']
for col in colunas_texto:
    if col in df_dicionario_silver.columns:
        df_dicionario_silver[col] = df_dicionario_silver[col].astype(str)

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_dicionario_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_dicionario_silver['_record_hash'] = df_dicionario_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/dicionario_silver.parquet"
df_dicionario_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade DICIONARIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_dicionario_silver)}")

 Processando entidade: DICIONARIO

Duplicatas removidas: 0

Valores nulos por coluna:
id_tabela               0
nome_coluna             0
chave                   0
cobertura_temporal      0
valor                   0
_ingestion_timestamp    0
_source_table           0
_source_system          0
_record_hash            0
dtype: int64

Tipos de dados atualizados:
id_tabela               str
nome_coluna             str
chave                   str
cobertura_temporal      str
valor                   str
_ingestion_timestamp    str
_source_table           str
_source_system          str
_record_hash            str
dtype: object
 Entidade DICIONARIO processada com sucesso na camada Silver!
 Salvo em: ./data_lake/silver/dicionario_silver.parquet
 Total de registros: 27


In [15]:
# ============================================================
# ETAPA 3.7 — TRANSFORMAÇÃO: ALUNOS
# ============================================================
print(f"{Fore.YELLOW} Processando entidade: ALUNOS\n")

# 1. Cópia do dataframe bronze
df_alunos_silver = bronze_dfs['alunos'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_alunos_silver)
df_alunos_silver = df_alunos_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_alunos_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_alunos_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
colunas_texto = ['id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado']
for col in colunas_texto:
    if col in df_alunos_silver.columns:
        df_alunos_silver[col] = df_alunos_silver[col].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['proficiencia', 'peso_aluno']
for col in colunas_numericas:
    if col in df_alunos_silver.columns:
        df_alunos_silver[col] = pd.to_numeric(df_alunos_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_alunos_silver.dtypes)

# 5. Atualização do hash de registro e salvamento na Silver
df_alunos_silver['_record_hash'] = df_alunos_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/alunos_silver.parquet"
df_alunos_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade ALUNOS processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_alunos_silver)}")

 Processando entidade: ALUNOS

Duplicatas removidas: 0

Valores nulos por coluna:
ano                           0
id_municipio                  0
id_escola                     0
id_aluno                      0
caderno                       0
serie                         0
rede                          0
presenca                      0
preenchimento_caderno         0
alfabetizado                  0
proficiencia             513338
peso_aluno               513338
_ingestion_timestamp          0
_source_table                 0
_source_system                0
_record_hash                  0
dtype: int64

Tipos de dados atualizados:
ano                        Int64
id_municipio                 str
id_escola                    str
id_aluno                     str
caderno                      str
serie                        str
rede                         str
presenca                     str
preenchimento_caderno        str
alfabetizado                 str
proficiencia             float64
p

# Etapa 4 - Camada GOLD: Agregações e Métricas de Negócio

In [16]:
# ============================================================
# ETAPA 4.1 — CONSOLIDAÇÃO MUNICIPIO E METAS
# ============================================================
print(f"{Fore.YELLOW} Processando consolidação Gold: Município e Metas\n")

# Dicionário para armazenar os dataframes da camada Gold
gold_dfs = {}

# 1. Carregar os dados limpos da Silver
df_municipio_silver = pd.read_parquet(f"{SILVER_PATH}/municipio_silver.parquet")
df_meta_municipio_silver = pd.read_parquet(f"{SILVER_PATH}/meta_alfabetizacao_municipio_silver.parquet")
df_dicionario_silver = pd.read_parquet(f"{SILVER_PATH}/dicionario_silver.parquet")

# 2. Criar o mapeamento usando a tabela de dicionário
df_map_rede = df_dicionario_silver[
    (df_dicionario_silver['id_tabela'] == 'municipio') & 
    (df_dicionario_silver['nome_coluna'] == 'rede')
][['chave', 'valor']].set_index('chave')['valor'].to_dict()

df_municipio_silver['rede_descricao'] = df_municipio_silver['rede'].map(df_map_rede)

# Na tabela de metas, a coluna de rede já está descritiva
df_meta_municipio_silver['rede_descricao'] = df_meta_municipio_silver['rede']

# 3. Realizando o inner join utilizando as descrições padronizadas
df_gold_municipio = pd.merge(
    df_municipio_silver,
    df_meta_municipio_silver,
    left_on=['ano', 'id_municipio', 'rede_descricao'],
    right_on=['ano', 'id_municipio', 'rede_descricao'],
    how='inner',
    suffixes=('_oficial', '_meta')
)

# 4. Limpeza de colunas auxiliares
df_gold_municipio = df_gold_municipio.drop(columns=['rede_descricao'])

# 5. Salvando na camada Gold
gold_dfs['municipio_metas'] = df_gold_municipio
output_file_gold = f"{GOLD_PATH}/municipio_metas_gold.parquet"
df_gold_municipio.to_parquet(output_file_gold, index=False)

print(f"{Fore.GREEN} Entidade MUNICIPIO_METAS processada com sucesso na camada Gold!")
print(f" Salvo em: {output_file_gold}")
print(f" Total de registros após o Inner Join: {len(df_gold_municipio)}")
print(f" Colunas resultantes: {list(df_gold_municipio.columns)}\n")

 Processando consolidação Gold: Município e Metas

 Entidade MUNICIPIO_METAS processada com sucesso na camada Gold!
 Salvo em: ./data_lake/gold/municipio_metas_gold.parquet
 Total de registros após o Inner Join: 10654
 Colunas resultantes: ['ano', 'id_municipio', 'serie', 'rede_oficial', 'taxa_alfabetizacao_oficial', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', '_ingestion_timestamp_oficial', '_source_table_oficial', '_source_system_oficial', '_record_hash_oficial', 'rede_meta', 'taxa_alfabetizacao_meta', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'nivel_alfabetizacao', 'percentual_participacao', '_ingestion_timestamp_meta', '_so

In [17]:
# ============================================================
# ETAPA 4.2 — CONSOLIDAÇÃO UF E METAS
# ============================================================
print(f"{Fore.YELLOW} Processando consolidação Gold: UF e Metas\n")

# 1. Carregar os dados limpos da Silver
df_uf_silver = pd.read_parquet(f"{SILVER_PATH}/uf_silver.parquet")
df_meta_uf_silver = pd.read_parquet(f"{SILVER_PATH}/meta_alfabetizacao_uf_silver.parquet")

# 2. Criar o mapeamento usando a tabela de dicionário
df_map_rede_uf = df_dicionario_silver[
    (df_dicionario_silver['id_tabela'] == 'uf') & 
    (df_dicionario_silver['nome_coluna'] == 'rede')
][['chave', 'valor']].set_index('chave')['valor'].to_dict()

# 3. Traduzir os códigos numéricos da UF para as descrições oficiais do dicionário
df_uf_silver['rede_descricao'] = df_uf_silver['rede'].map(df_map_rede_uf)

# 4. Padronização semântica robusta: 
# Se a descrição contiver a palavra "Pública", unificamos para "Pública" 
# para alinhar perfeitamente com a granularidade da tabela de metas, que tem sua descrição 
# diferente de como é na tabela de dicionário
df_uf_silver['rede_descricao'] = df_uf_silver['rede_descricao'].apply(
    lambda x: 'Pública' if isinstance(x, str) and 'Pública' in x else x
)

# Na tabela de metas, a coluna de rede já vem como 'Pública'
df_meta_uf_silver['rede_descricao'] = df_meta_uf_silver['rede']

# 5. Realizando o inner join utilizando as descrições padronizadas
df_gold_uf = pd.merge(
    df_uf_silver,
    df_meta_uf_silver,
    left_on=['ano', 'sigla_uf', 'rede_descricao'],
    right_on=['ano', 'sigla_uf', 'rede_descricao'],
    how='inner',
    suffixes=('_oficial', '_meta')
)

# 6. Limpeza de colunas auxiliares
df_gold_uf = df_gold_uf.drop(columns=['rede_descricao'])

# 7. Salvando na camada Gold
gold_dfs['uf_metas'] = df_gold_uf
output_file_gold_uf = f"{GOLD_PATH}/uf_metas_gold.parquet"
df_gold_uf.to_parquet(output_file_gold_uf, index=False)

print(f"{Fore.GREEN} Entidade UF_METAS processada com sucesso na camada Gold!")
print(f" Salvo em: {output_file_gold_uf}")
print(f" Total de registros após o Inner Join: {len(df_gold_uf)}")
print(f" Colunas resultantes: {list(df_gold_uf.columns)}\n")

 Processando consolidação Gold: UF e Metas

 Entidade UF_METAS processada com sucesso na camada Gold!
 Salvo em: ./data_lake/gold/uf_metas_gold.parquet
 Total de registros após o Inner Join: 49
 Colunas resultantes: ['ano', 'sigla_uf', 'serie', 'rede_oficial', 'taxa_alfabetizacao_oficial', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', '_ingestion_timestamp_oficial', '_source_table_oficial', '_source_system_oficial', '_record_hash_oficial', 'rede_meta', 'taxa_alfabetizacao_meta', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'percentual_participacao', '_ingestion_timestamp_meta', '_source_table_meta', '_source_system_meta', '_record_h

In [18]:
# ============================================================
# ETAPA 4.3 — ENRIQUECIMENTO DINÂMICO DA TABELA ALUNOS
# ============================================================
print(f"{Fore.YELLOW} Processando consolidação Gold: Alunos\n")

# 1. Carregar os dados limpos da Silver
df_alunos_silver = pd.read_parquet(f"{SILVER_PATH}/alunos_silver.parquet")

# 2. Descobrir dinamicamente quais colunas da tabela 'alunos' possuem mapeamento no dicionário
colunas_para_mapear = df_dicionario_silver[df_dicionario_silver['id_tabela'] == 'alunos']['nome_coluna'].unique()

print(f"Colunas identificadas dinamicamente para tradução: {list(colunas_para_mapear)}")

# 3. Função de mapeamento genérica e dinâmica
df_gold_alunos = df_alunos_silver.copy()

for coluna in colunas_para_mapear:
    # Extrai o dicionário (chave -> valor) para a coluna atual
    mapa = df_dicionario_silver[
        (df_dicionario_silver['id_tabela'] == 'alunos') & 
        (df_dicionario_silver['nome_coluna'] == coluna)
    ][['chave', 'valor']].set_index('chave')['valor'].to_dict()
    
    if mapa and coluna in df_gold_alunos.columns:
        coluna_destino = f"{coluna}_descricao"
        df_gold_alunos[coluna_destino] = df_gold_alunos[coluna].map(mapa)

# 4. Salvando na camada Gold
gold_dfs['alunos'] = df_gold_alunos
output_file_gold_alunos = f"{GOLD_PATH}/alunos_gold.parquet"
df_gold_alunos.to_parquet(output_file_gold_alunos, index=False)

print(f"{Fore.GREEN} Entidade ALUNOS processada com sucesso na camada Gold!")
print(f" Salvo em: {output_file_gold_alunos}")
print(f" Total de registros: {len(df_gold_alunos)}")
print(f" Colunas resultantes: {list(df_gold_alunos.columns)}\n")

 Processando consolidação Gold: Alunos

Colunas identificadas dinamicamente para tradução: ['alfabetizado', 'preenchimento_caderno', 'presenca', 'rede', 'serie']
 Entidade ALUNOS processada com sucesso na camada Gold!
 Salvo em: ./data_lake/gold/alunos_gold.parquet
 Total de registros: 3867999
 Colunas resultantes: ['ano', 'id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado', 'proficiencia', 'peso_aluno', '_ingestion_timestamp', '_source_table', '_source_system', '_record_hash', 'alfabetizado_descricao', 'preenchimento_caderno_descricao', 'presenca_descricao', 'rede_descricao', 'serie_descricao']

